# Fine-tuning a foundation model: PET-MAD-XS on ethanol

A **foundation** (or *universal*) machine-learning interatomic potential is trained once, on
a broad and chemically diverse dataset, and is then meant to describe essentially any
system without being re-fitted. That is a remarkable promise — but "describes any system"
and "describes *your* system to the accuracy you need" are two different statements, and
the gap between them is what this notebook is about.

We take **PET-MAD-XS**, a foundation model trained on DFT data spanning 102 elements, and
put it in front of a molecule it was never specifically fitted to: ethanol, with
**CCSD(T)** reference energies and forces — a much higher level of theory than the DFT the
model was trained on. We then ask three questions, in increasing order of difficulty:

1. Does it get the **forces** right? (easy — it is a good model)
2. Does it get the **structure** right, i.e. bond lengths and angles sampled by molecular
   dynamics? (still fairly easy)
3. Does it get the **conformational energy landscape** right — the few-meV energy
   difference between the *anti* and *gauche* forms of ethanol? (hard, and the honest
   answer is: not zero-shot)

Between question 2 and question 3 we **fine-tune** the model on a few hundred CCSD(T)
structures, which takes a couple of minutes, and repeat everything.

### Roadmap

| § | What we do |
|---|---|
| 1 | Look at the molecule and the CCSD(T) reference dataset |
| 2 | Split the data into train / validation / test |
| 3 | Load PET-MAD-XS zero-shot |
| 4 | Zero-shot accuracy on held-out CCSD(T) energies and forces |
| 5 | Relax, then run molecular dynamics — including a hands-on look at the timestep |
| 6 | Compare MD structure against the CCSD(T) reference |
| 7 | Fine-tune PET-MAD-XS |
| 8 | Repeat §4-§6 with the fine-tuned model: how much did it help? |
| 9 | The hard test: the C1-C2-O-H torsional potential and the anti/gauche balance |

### How to work through this

Cells marked **Your turn** contain a `TODO`: a parameter to choose, a prediction to make
before running the next cell, or a number to read off a plot. They are the point of the
session — the notebook runs fine with the defaults, but you will learn much more if you
change them and watch what happens.

**Timing.** Single-point predictions and minimisations finish in seconds. Each MD run
takes a few minutes, the timestep experiment about a minute, and fine-tuning two to three
minutes on a GPU (longer on CPU). If you are short on time, turn `nsteps` down in §5 —
everything downstream still works, just with noisier histograms.

## Setup

Two files are all we need. Neither is in this repository: both sit in shared directories on
the classroom machine, so that fifty people are not each carrying their own copy. Running
elsewhere, point the two paths in the next cell at your own copies.

* `/home/unito/dataset/ethanol_ccsd_t.xyz` — 2,000 ethanol geometries with energies and
  forces at the **CCSD(T)/cc-pVTZ** level, converted from the original
  [sGDML / MD17](http://www.sgdml.org/#datasets) release into extended XYZ with everything
  in eV and eV/A.
* `/home/unito/pet-mad-models/pet-mad-xs-v1.6.0.ckpt` — the PET-MAD-XS
  checkpoint. It lives outside the repository, on the classroom machine, and is shared
  with the `water-md` notebook. If you are running elsewhere, point `CKPT_PATH` below at
  your own copy.

Everything runs on a GPU if one is visible, and falls back to CPU otherwise.

In [ ]:
import subprocess
import sys
import tempfile
import time
import warnings
from pathlib import Path

import ase.io
import chemiscope
import matplotlib.pyplot as plt
import numpy as np
import torch
from ase import units
from ase.optimize import LBFGS
from metatomic_ase import MetatomicCalculator
from tqdm.auto import tqdm

# ethanol has no periodic cell, so the stress ASE tries to compute along the way is
# meaningless here; these warnings are harmless, we just don't want 50 people worrying
# about them at once.
warnings.filterwarnings("ignore", category=RuntimeWarning, module="metatomic_ase")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using device: {DEVICE}")

DATA_PATH = Path("/home/unito/dataset/ethanol_ccsd_t.xyz")
CKPT_PATH = Path("/home/unito/pet-mad-models/pet-mad-xs-v1.6.0.ckpt")
MODEL_PATH = Path("pet-mad-xs.pt")

rng = np.random.default_rng(0)

### What is actually inside the checkpoint?

Before using the model as a black box, it costs nothing to look at what it is. The
checkpoint stores the architecture hyperparameters next to the weights, so we can read off
the two numbers that matter most for the physics — the **cutoff radius** (how far an atom
is allowed to "see") and the **number of message-passing layers** (how far information
travels beyond that, one cutoff per layer) — plus the size of the model and how many
chemical elements it claims to cover.

In [ ]:
import metatomic.torch  # noqa: F401  registers the classes referenced by the checkpoint

_ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
hypers = _ckpt["model_data"]["model_hypers"]
info = _ckpt["model_data"]["dataset_info"]


def _count_params(obj):
    if torch.is_tensor(obj):
        return obj.numel()
    if isinstance(obj, dict):
        return sum(_count_params(v) for v in obj.values())
    if isinstance(obj, (list, tuple)):
        return sum(_count_params(v) for v in obj)
    return 0


print(f"architecture          : {_ckpt['architecture_name'].upper()}")
print(f"parameters            : {_count_params(_ckpt['model_state_dict']) / 1e6:.2f} M")
print(f"cutoff radius         : {hypers['cutoff']} A")
print(f"message-passing layers: {hypers['num_gnn_layers']}  "
      f"(effective range ~{hypers['cutoff'] * hypers['num_gnn_layers']:.0f} A)")
print(f"attention heads       : {hypers['num_heads']}, embedding width {hypers['d_pet']}")
print(f"elements covered      : {len(info.atomic_types)}")

del _ckpt  # we only needed the metadata

Two things are worth noticing. First, this is a **small** model by machine-learning
standards — a few million parameters, comparable to an image classifier from 2014, not to
a language model. Second, the 7.5 A cutoff is *large* for a molecule this size: every atom
in ethanol sees every other atom, so the whole molecule is inside one environment. That
matters later — the model has all the information it needs to distinguish conformers; if
it gets their energy ordering wrong, it is not because it cannot see the difference.

## 1. A first look at the data

The reference dataset is not a set of optimised structures: it is 2,000 snapshots sampled
along a high-temperature (500 K) *ab initio* molecular dynamics trajectory, each one
recomputed at the CCSD(T) level. So it is a **thermal** sample — bonds are stretched and
compressed, the methyl and hydroxyl groups have rotated, and the total energies span more
than an electronvolt.

That is exactly what you want for fitting a potential: a model trained only on minima
knows nothing about the forces that push a molecule back towards them.

In [ ]:
frames = ase.io.read(DATA_PATH, ":")
energies = np.array([atoms.get_potential_energy() for atoms in frames])

print(f"{len(frames)} reference structures, {len(frames[0])} atoms each "
      f"({frames[0].get_chemical_formula()})")
print(f"energies span {energies.min():.3f} to {energies.max():.3f} eV "
      f"(a range of {1000 * np.ptp(energies):.0f} meV)")
print(f"largest force component: {max(np.abs(a.get_forces()).max() for a in frames):.2f} eV/A")

fig, ax = plt.subplots(figsize=(6, 3), dpi=120)
ax.hist(1000 * (energies - energies.min()), bins=60, color="tab:grey")
ax.set(xlabel="CCSD(T) energy above the lowest sampled structure (meV)",
       ylabel="structures", title="A thermal sample, not a set of minima")
fig.tight_layout()
plt.show()

chemiscope.show(structures=frames[:1], mode="structure")

Rotate the molecule in the viewer above: a methyl group (CH$_3$), a methylene group
(CH$_2$) and a hydroxyl group (OH), all in a chain, C-C-O-H.

The cell below finds those atoms automatically from the connectivity, so that nothing
downstream depends on the order the atoms happen to be listed in. We give the two carbons
the names we will use throughout: **C1** is the methyl carbon and **C2** is the carbon
bonded to the oxygen, so the chain we care about in §9 is **C1-C2-O-H**.

In [ ]:
# Identify the skeleton from connectivity (bond = shorter than 1.3x the sum of covalent
# radii). This only needs to run once: ethanol's covalent bonds never break during the
# short, room-temperature MD trajectories we run below.
covalent_radius = {"C": 0.76, "O": 0.66, "H": 0.31}
symbols = np.array(frames[0].get_chemical_symbols())
distances = frames[0].get_all_distances()
n_atoms = len(frames[0])

neighbors = {i: [] for i in range(n_atoms)}
for i in range(n_atoms):
    for j in range(i + 1, n_atoms):
        cutoff = 1.3 * (covalent_radius[symbols[i]] + covalent_radius[symbols[j]])
        if distances[i, j] < cutoff:
            neighbors[i].append(j)
            neighbors[j].append(i)

oxygen = int(np.where(symbols == "O")[0][0])
hydroxyl_h = [j for j in neighbors[oxygen] if symbols[j] == "H"][0]
ch2_carbon = [j for j in neighbors[oxygen] if symbols[j] == "C"][0]   # "C2"
methyl_carbon = [j for j in neighbors[ch2_carbon] if symbols[j] == "C"][0]  # "C1"

skeleton = {
    "C1(CH3)": methyl_carbon,
    "C2(CH2)": ch2_carbon,
    "O": oxygen,
    "H(hydroxyl)": hydroxyl_h,
}
# the C1-C2-O-H dihedral: the hydroxyl internal rotation, and the star of section 9
TORSION = (methyl_carbon, ch2_carbon, oxygen, hydroxyl_h)

print("Skeleton atom indices:", skeleton)
print("C1-C2-O-H torsion in the first frame: "
      f"{frames[0].get_dihedral(*TORSION):.1f} degrees")

## 2. Split the data into train / validation / test

Three disjoint sets, each with a different job:

* **training** — the structures the fine-tuning actually learns from;
* **validation** — never trained on, watched during training to detect overfitting and to
  pick the best epoch;
* **test** — touched only once, at the very end, to quote an honest accuracy number.

> **Your turn.** Choose the three sizes in the cell below. The default (500 / 100 / 100) is
> a good compromise for a live session: fine-tuning takes about two minutes. Try 100
> training structures instead and watch the validation error in §7 — how little data can
> you get away with? Note that the three numbers must add up to at most 2,000.

One honest caveat about this split, worth thinking about: our 2,000 structures come from a
*single continuous MD trajectory*, so consecutive frames are highly correlated. A random
split therefore puts near-copies of training structures into the test set, and the test
error it reports is optimistic compared to what the model would do on a genuinely new
molecule or a different temperature. Splitting by time (first 80% train, last 20% test)
would be stricter. We keep the random split because it is the standard choice and the
comparison we care about — zero-shot versus fine-tuned, on identical structures — is fair
either way.

In [ ]:
n_train = 1600  # TODO: how many structures to fine-tune on? (try 100, 200, 1000, ...)
n_val = 200    # TODO: how many to validate on while fine-tuning?
n_test = 200   # TODO: how many to hold out for the final, honest accuracy check?

assert n_train + n_val + n_test <= len(frames), "not enough structures for that split!"

shuffled = rng.permutation(len(frames))
train_idx = shuffled[:n_train]
val_idx = shuffled[n_train:n_train + n_val]
test_idx = shuffled[n_train + n_val:n_train + n_val + n_test]

# the three splits live in memory and nowhere else; §7 gives `metatrain` a temporary copy
# of the two it needs, and cleans up after itself
train_frames = [frames[i] for i in train_idx]
val_frames = [frames[i] for i in val_idx]
test_frames = [frames[i] for i in test_idx]

print(f"train: {len(train_frames)}, val: {len(val_frames)}, test: {len(test_frames)}")

## 3. Load PET-MAD-XS zero-shot

`metatrain export` compiles the checkpoint into a TorchScript file, `pet-mad-xs.pt`, which is
what both ASE and LAMMPS load. It is written once and reused by every cell below.

`MetatomicCalculator` then makes that file look like any other ASE calculator: attach it
to an `Atoms` object and `get_potential_energy()` / `get_forces()` work as usual.

In [ ]:
if not MODEL_PATH.exists():
    subprocess.run(
        [sys.executable, "-m", "metatrain", "export", str(CKPT_PATH), "-o", str(MODEL_PATH)],
        check=True,
    )

zero_shot_calculator = MetatomicCalculator(str(MODEL_PATH), device=DEVICE)
print(f"model ready at {MODEL_PATH}")

## 4. How good is PET-MAD-XS out of the box?

Now the first real measurement. One subtlety has to be dealt with first.

**Energies are only defined up to a constant, and the two conventions differ.** PET-MAD-XS
predicts energies in the reference frame of its DFT training data; our dataset is at
CCSD(T) level with its own zero. The difference is a constant *for a fixed composition*,
and every structure here is the same C$_2$H$_6$O molecule — so a single number, fitted on
a handful of training structures, removes it completely. We measure it below rather than
assume it.

**Forces need no such treatment**: they are derivatives of the energy, and the constant
differentiates away. This is why forces are the more meaningful zero-shot benchmark, and
also why they are what MD actually consumes.

> **Your turn.** Before running the cell: PET-MAD-XS has never seen a CCSD(T) ethanol
> structure, and was trained on DFT. Guess the force MAE you expect, in meV/A. For
> reference, a *good* system-specific potential fitted to this dataset would be around
> 10 meV/A, and the forces here reach several eV/A.

In [ ]:
def predict(calculator, atoms_list, desc="evaluating"):
    # energies (eV) and forces (eV/A) predicted for a list of structures
    energies, forces = [], []
    for atoms in tqdm(atoms_list, desc=desc, leave=False):
        probe = atoms.copy()
        probe.calc = calculator
        energies.append(probe.get_potential_energy())
        forces.append(probe.get_forces())
    return np.array(energies), np.array(forces)


def reference(atoms_list):
    return (np.array([a.get_potential_energy() for a in atoms_list]),
            np.array([a.get_forces() for a in atoms_list]))


# the constant offset between the two energy conventions, fitted on a few training
# structures only -- one constant needs very little data
n_shift = min(100, len(train_frames))
shift_pred_e, _ = predict(zero_shot_calculator, train_frames[:n_shift], "fitting the offset")
shift_ref_e, _ = reference(train_frames[:n_shift])
energy_shift = np.mean(shift_ref_e - shift_pred_e)
print(f"energy offset between the DFT and CCSD(T) conventions: {energy_shift:.3f} eV "
      f"({1000 * energy_shift / n_atoms:.0f} meV/atom)")

zs_test_e, zs_test_f = predict(zero_shot_calculator, test_frames, "zero-shot test set")
ref_test_e, ref_test_f = reference(test_frames)

zs_energy_mae = 1000 * np.mean(np.abs(zs_test_e + energy_shift - ref_test_e)) / n_atoms
zs_forces_mae = 1000 * np.mean(np.abs(zs_test_f - ref_test_f))
print(f"zero-shot energy MAE: {zs_energy_mae:.2f} meV/atom")
print(f"zero-shot force  MAE: {zs_forces_mae:.1f} meV/A")

In [ ]:
def parity_plots(pred_e, pred_f, label, color, energy_mae, forces_mae):
    fig, axes = plt.subplots(1, 2, figsize=(9, 4), dpi=120)

    axes[0].scatter(ref_test_e, pred_e, s=12, alpha=0.6, color=color)
    lims = [ref_test_e.min(), ref_test_e.max()]
    axes[0].plot(lims, lims, "k--", lw=1)
    axes[0].set(xlabel="CCSD(T) energy (eV)", ylabel=f"{label} energy (eV)",
                title=f"Energy MAE: {energy_mae:.2f} meV/atom")

    axes[1].scatter(ref_test_f.ravel(), pred_f.ravel(), s=4, alpha=0.2, color=color)
    flims = [ref_test_f.min(), ref_test_f.max()]
    axes[1].plot(flims, flims, "k--", lw=1)
    axes[1].set(xlabel="CCSD(T) force component (eV/A)",
                ylabel=f"{label} force component (eV/A)",
                title=f"Force MAE: {forces_mae:.1f} meV/A")
    fig.suptitle(f"{label} vs CCSD(T) on the held-out test set", fontsize=11)
    fig.tight_layout()
    plt.show()


parity_plots(zs_test_e + energy_shift, zs_test_f, "PET-MAD-XS zero-shot", "tab:blue",
             zs_energy_mae, zs_forces_mae)

This is a good result for a model that has never seen this molecule at this level of
theory: the points sit on the diagonal, and both errors are small compared to the range
being predicted. Keep the two numbers in mind — we will beat them in §8, and in §9 we will
find an observable that is *smaller* than the energy error printed above, which is where
the trouble starts.

## 5. Relax, then run molecular dynamics

Accuracy on a fixed set of structures is one thing; what we actually want a potential for
is to *generate* structures. So we now let the model drive the molecule itself.

First a **geometry optimisation** (LBFGS), which walks downhill in energy until the forces
vanish, landing in the model's nearest local minimum. This is the model's idea of the
equilibrium structure of ethanol.

In [ ]:
def relax(atoms, calculator, fmax=0.01, steps=300, keep_trajectory=False):
    # minimise the energy until every force component is below fmax (eV/A)
    atoms = atoms.copy()
    atoms.calc = calculator
    trajectory = [atoms.copy()]
    opt = LBFGS(atoms, logfile=None)
    if keep_trajectory:
        opt.attach(lambda: trajectory.append(atoms.copy()))
    opt.run(fmax=fmax, steps=steps)
    print(f"relaxed in {opt.nsteps} steps, "
          f"max force = {np.abs(atoms.get_forces()).max():.4f} eV/A")
    return (atoms, trajectory) if keep_trajectory else atoms


zs_relaxed, relax_trajectory = relax(frames[0], zero_shot_calculator, keep_trajectory=True)

Watch the geometry settle into the model's local minimum — the last frame is `zs_relaxed`,
the starting point for the dynamics below:

In [ ]:
chemiscope.show(
    structures=relax_trajectory,
    mode="structure",
    settings=chemiscope.quick_settings(
        trajectory=True,
        structure_settings={"playbackDelay": 100},
    ),
)

### The dynamics

**A little bit of theory.** Molecular dynamics integrates Newton's equations of motion step
by step. ASE, like almost every MD code, uses the **velocity Verlet** algorithm: given
positions $x_t$, velocities $v_t$ and forces $F_t = -\nabla U(x_t)$,

$$x_{t+\Delta t} = x_t + v_t \Delta t + \tfrac{1}{2} \tfrac{F_t}{m} \Delta t^2, \qquad
v_{t+\Delta t} = v_t + \tfrac{1}{2}\tfrac{F_t + F_{t+\Delta t}}{m} \Delta t.$$

It costs one force evaluation per step — which here is one neural-network forward pass,
i.e. essentially the entire cost of the simulation — and it is *time-reversible* and
*symplectic*, which is why it conserves energy so well over long runs.

We run in the **NVE** ensemble: no thermostat, constant total energy. That makes the
simulation an honest test of the integrator, because the total energy is then a conserved
quantity that we can watch for drift — which is exactly the experiment in the next cell.

**Why we start at twice the target temperature.** We begin at the potential-energy
*minimum*, where all the energy we give the molecule is kinetic. As it starts to vibrate,
equipartition splits that energy roughly evenly between kinetic and potential — so a run
launched with $T$ worth of velocities settles at about $T/2$. Seeding at $2T$ lands us
near $T$; the rule is only approximate, because real vibrations are anharmonic, so what
counts is the measured average we print below, not the number we asked for.

We also strip the net translation and rotation, which carry kinetic energy without
contributing to any internal motion. That leaves $3N-6 = 21$ vibrational degrees of
freedom, and the temperature has to be computed with that number rather than $3N$.

**Which temperature?** We run at **500 K**, because that is the temperature the reference
trajectory was sampled at. Comparing distributions computed at different temperatures is a
classic way to manufacture a disagreement that is not the model's fault — bonds and angles
are anharmonic, so their *average* values shift with temperature even for a perfect
potential.

In [ ]:
from ase.md.velocitydistribution import Stationary, ZeroRotation, thermalize_momenta
from ase.md.verlet import VelocityVerlet

NDOF = 3 * n_atoms - 6  # 3N minus translations and rotations of a non-linear molecule


def temperature_of(atoms_or_kinetic):
    # instantaneous temperature from the vibrational degrees of freedom only
    ke = (atoms_or_kinetic.get_kinetic_energy()
          if hasattr(atoms_or_kinetic, "get_kinetic_energy") else atoms_or_kinetic)
    return 2 * ke / (NDOF * units.kB)


def run_md(calculator, start_atoms, nsteps=4000, dt_fs=0.5, temperature_K=500, seed=0,
           desc="MD", progress=True):
    atoms = start_atoms.copy()
    atoms.calc = calculator
    rng_local = np.random.default_rng(seed)

    thermalize_momenta(atoms, 2 * temperature_K, rng=rng_local)  # see the note above
    Stationary(atoms)    # remove net translation
    ZeroRotation(atoms)  # remove net rotation

    dyn = VelocityVerlet(atoms, dt_fs * units.fs)

    trajectory, kinetic, potential = [], [], []

    def collect():
        trajectory.append(atoms.copy())
        kinetic.append(atoms.get_kinetic_energy())
        potential.append(atoms.get_potential_energy())

    dyn.attach(collect, interval=1)
    bar = tqdm(total=nsteps, desc=desc, leave=False) if progress else None
    if bar is not None:
        dyn.attach(lambda: bar.update(1), interval=1)
    dyn.run(nsteps)
    if bar is not None:
        bar.close()

    return trajectory, np.array(kinetic), np.array(potential)

### Your turn: how long a timestep can you get away with?

The timestep is on of the most important knob in any MD simulation, and it is a pure
trade-off: doubling $\Delta t$ halves the cost of simulating a given amount of physical
time, but the integration error grows. Velocity Verlet's energy error per step scales as
$\Delta t^2$, so the total energy of an NVE run stops being constant — it *fluctuates*,
and eventually it *drifts*.

The rule of thumb is that $\Delta t$ must resolve the fastest vibration in the system.
Here that is the C-H stretch at about 3000 cm$^{-1}$, i.e. a period of roughly **11 fs**.

> **Your turn.** Before running the cell: how many timesteps per period do you think you
> need? Write down the largest $\Delta t$ you would trust. Then run the cell — it launches
> a short NVE run for each timestep in `TIMESTEPS_FS` — and compare.

In [ ]:
TIMESTEPS_FS = [0.5, 1.0, 2.0, 4.0]  # TODO: add 0.25 (slow!) or try 3.0, 5.0
TEST_PS = 0.5  # physical time simulated for each timestep, so the comparison is fair

drift = {}
for dt in TIMESTEPS_FS:
    n_steps_dt = int(1000 * TEST_PS / dt)
    _, kin, pot = run_md(zero_shot_calculator, zs_relaxed, nsteps=n_steps_dt, dt_fs=dt,
                         seed=1, desc=f"dt = {dt} fs")
    total = kin + pot
    drift[dt] = (np.arange(len(total)) * dt / 1000, 1000 * (total - total[0]))
    print(f"dt = {dt:4.2f} fs ({n_steps_dt:5d} steps): "
          f"largest |E_tot - E_tot(0)| = {np.abs(drift[dt][1]).max():12.2f} meV")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=120)

for dt, (t_ps, dE) in drift.items():
    axes[0].plot(t_ps, np.abs(dE) + 1e-3, lw=1, label=f"{dt} fs")
axes[0].set(xlabel="time (ps)", ylabel="|E$_{tot}$(t) - E$_{tot}$(0)| (meV)", yscale="log",
            title="Total energy should be constant in NVE")
axes[0].legend(title="timestep")

dts = np.array(sorted(drift))
worst = np.array([np.abs(drift[dt][1]).max() for dt in dts])
axes[1].loglog(dts, worst, "o-", color="tab:red")
guide = worst[0] * (dts / dts[0]) ** 2
axes[1].loglog(dts, guide, "k:", lw=1, label=r"$\propto \Delta t^2$")
axes[1].set(xlabel="timestep (fs)", ylabel="largest energy error (meV)",
            title="Error growth with timestep")
axes[1].legend()

fig.tight_layout()
plt.show()

Read the numbers against that 11 fs period, in steps per oscillation:

* **0.5 fs — about 22 steps per period.** The total energy wobbles a little and the wobble
  is *bounded*: it oscillates rather than growing. That boundedness is the symplectic
  property of Verlet at work, and it is why we can run for millions of steps without the
  energy wandering off.
* **1 fs — about 11 steps per period.** Still stable, and the error grows by roughly the
  factor of four that the $\Delta t^2$ scaling predicts (right-hand panel). Usable — but
  note that the fluctuation is now the same size as the energy differences §9 is about.
* **2 fs and beyond — fewer than 6 steps per period.** The integrator stops resolving the
  C-H stretch altogether and the energy runs away: several eV at 2 fs, and utterly absurd
  values by 4 fs. The molecule is effectively blown apart. This is not an inaccurate
  simulation, it is a meaningless one — and it is glaringly obvious in the diagnostic,
  which is exactly why you always plot it.

Two warnings worth taking away. First, the $\Delta t^2$ guide line only describes the
*stable* runs; once the integrator goes unstable there is no scaling law, just an
explosion. Second, the threshold is not sharp: near it, whether a run survives depends on
the initial velocities, so the same timestep may work in one trajectory and destroy the
next. Marginal stability is not stability.

This is why MD with hydrogen usually runs at 0.5-1 fs. (The usual escape routes are to
constrain the X-H bonds, or to give hydrogen a heavier mass, both of which slow the fastest
motion down so that a longer timestep becomes safe.)

> **Your turn.** Set `MD_TIMESTEP_FS` below to the value *you* would use for a production
> run, then run the real trajectory.

In [ ]:
MD_TIMESTEP_FS = 0.5    # TODO: your choice, informed by the experiment above
MD_STEPS = 4000         # TODO: total steps; 4000 x 0.5 fs = 2 ps. Fewer = faster, noisier
MD_TEMPERATURE_K = 500  # matches the temperature the reference dataset was sampled at
EQUILIBRATION = 500     # frames discarded before averaging (see the plot below)

t0 = time.time()
zs_traj, zs_kinetic, zs_potential = run_md(
    zero_shot_calculator, zs_relaxed, nsteps=MD_STEPS+EQUILIBRATION, dt_fs=MD_TIMESTEP_FS,
    temperature_K=MD_TEMPERATURE_K, desc="zero-shot MD")
print(f"zero-shot MD: {len(zs_traj)} frames "
      f"({MD_STEPS * MD_TIMESTEP_FS / 1000:.1f} ps) in {time.time() - t0:.1f} s")

The plot below is the standard sanity check for any MD run. The potential and kinetic
energies exchange energy back and forth continuously — that *is* the vibration — while
their sum should be a flat line. The first few hundred femtoseconds, while the initial
kinetic energy redistributes, are not representative of equilibrium and are discarded
before any averaging (red dotted line).

The temperature panel shows why a single snapshot never tells you the temperature of a
small system: for 21 vibrational degrees of freedom the instantaneous value swings by
+-100 K or so. Relative fluctuations scale as $1/\sqrt{N_\text{dof}}$, which for one
small molecule is a large number. Only the *average* is meaningful.

In [ ]:
def plot_md_diagnostics(kinetic, potential, dt_fs, label):
    t_ps = np.arange(len(kinetic)) * dt_fs / 1000
    total = kinetic + potential
    temperature = np.array([temperature_of(k) for k in kinetic])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=120)

    axes[0].plot(t_ps, 1000 * (potential - potential[0]), lw=0.8, color="tab:blue",
                 label="potential")
    axes[0].plot(t_ps, 1000 * kinetic, lw=0.8, color="tab:orange", label="kinetic")
    axes[0].plot(t_ps, 1000 * (total - total[0]), lw=1.2, color="tab:green",
                 label="total (should be flat)")
    axes[0].axvline(EQUILIBRATION * dt_fs / 1000, color="tab:red", ls=":", lw=1,
                    label="end of equilibration")
    axes[0].set(xlabel="time (ps)", ylabel="energy relative to the first frame (meV)",
                title=f"Energy components - {label}")
    axes[0].legend(fontsize=8)

    axes[1].plot(t_ps, temperature, lw=0.6, color="tab:grey")
    mean_T = temperature[EQUILIBRATION:].mean()
    axes[1].axhline(mean_T, color="tab:red", lw=1.5,
                    label=f"mean after equilibration: {mean_T:.0f} K")
    axes[1].axvline(EQUILIBRATION * dt_fs / 1000, color="tab:red", ls=":", lw=1)
    axes[1].set(xlabel="time (ps)", ylabel="instantaneous temperature (K)",
                title=f"Temperature ({NDOF} vibrational dof) - {label}")
    axes[1].legend(fontsize=8)

    fig.tight_layout()
    plt.show()
    print(f"{label}: mean temperature after equilibration = {mean_T:.0f} K, "
          f"total-energy drift over the run = "
          f"{1000 * (total[-1] - total[0]):.1f} meV")


plot_md_diagnostics(zs_kinetic, zs_potential, MD_TIMESTEP_FS, "zero-shot")

Let us watch the molecule move before analysing it quantitatively:

In [ ]:
chemiscope.show(
    structures=zs_traj[::10],
    mode="structure",
    settings=chemiscope.quick_settings(trajectory=True),
)

## 6. Does the dynamics reproduce the CCSD(T) structure?

Now the quantitative comparison. We measure three bond lengths and two angles along the
trajectory and compare their *distributions* to the same quantities computed directly from
the CCSD(T) reference structures — which costs nothing, since the reference geometries are
already on disk.

Because we deliberately ran at the reference dataset's own temperature, this is a
like-for-like comparison: both distributions sample the same ensemble, so both their
**widths** and their **centres** should agree if the model is right.

<img src="pics/ethanol_bonds.png" width=400 height=300 />

In [ ]:
BOND_COLORS = {"C1-C2": "tab:blue", "C2-O": "tab:green", "O-H": "tab:red"}
ANGLE_COLORS = {"C1-C2-O": "tab:purple", "C2-O-H": "tab:orange"}

def bond_lengths(atoms_list):
    pairs = {"C1-C2": (skeleton["C1(CH3)"], skeleton["C2(CH2)"]),
             "C2-O": (skeleton["C2(CH2)"], skeleton["O"]),
             "O-H": (skeleton["O"], skeleton["H(hydroxyl)"])}
    return {label: np.array([a.get_distance(i, j) for a in atoms_list])
            for label, (i, j) in pairs.items()}


def bond_angles(atoms_list):
    triples = {"C1-C2-O": (skeleton["C1(CH3)"], skeleton["C2(CH2)"], skeleton["O"]),
               "C2-O-H": (skeleton["C2(CH2)"], skeleton["O"], skeleton["H(hydroxyl)"])}
    return {label: np.array([a.get_angle(i, j, k) for a in atoms_list])
            for label, (i, j, k) in triples.items()}


# the CCSD(T) reference distributions, straight from the dataset
ref_bonds = bond_lengths(frames)
ref_angles = bond_angles(frames)

zs_bonds = bond_lengths(zs_traj[EQUILIBRATION:])
zs_angles = bond_angles(zs_traj[EQUILIBRATION:])


def compare_structure(md_results):
    # md_results: {label: (bonds, angles, linestyle)}
    fig, axes = plt.subplots(1, 2, figsize=(13, 4), dpi=120)

    for label, color in BOND_COLORS.items():
        axes[0].hist(ref_bonds[label], bins=60, density=True, histtype="step", lw=1.6,
                     color=color, label=f"{label}  CCSD(T) reference")
    for name, (bonds, _, ls) in md_results.items():
        for label, color in BOND_COLORS.items():
            axes[0].axvline(bonds[label].mean(), color=color, ls=ls, lw=2)
    axes[0].set(xlabel="bond length (A)", ylabel="probability density",
                title="Bonds: outline = CCSD(T) at 500 K, vertical lines = MD means")
    axes[0].legend(fontsize=7)

    for label, color in ANGLE_COLORS.items():
        axes[1].hist(ref_angles[label], bins=60, density=True, histtype="step", lw=1.6,
                     color=color, label=f"{label}  CCSD(T) reference")
    for name, (_, angles_, ls) in md_results.items():
        for label, color in ANGLE_COLORS.items():
            axes[1].axvline(angles_[label].mean(), color=color, ls=ls, lw=2)
    axes[1].set(xlabel="angle (degrees)", ylabel="probability density",
                title="Angles: outline = CCSD(T), vertical lines = MD means")
    axes[1].legend(fontsize=7)

    style_handles = [plt.Line2D([0], [0], color="k", ls=ls, lw=2, label=name)
                     for name, (_, _, ls) in md_results.items()]
    axes[0].add_artist(axes[0].get_legend())  # keep the per-bond legend ...
    axes[0].legend(handles=style_handles, fontsize=7, loc="upper center")  # ... and add this one
    fig.tight_layout()
    plt.show()

    def blocked_error(values, n_blocks=5):
        # MD frames are strongly correlated, so the naive standard error is far too
        # optimistic. Averaging within blocks first gives an honest estimate.
        blocks = np.array_split(values, n_blocks)
        return np.std([b.mean() for b in blocks], ddof=1) / np.sqrt(n_blocks)

    header = f"{'observable':>10} | {'CCSD(T)':>10} |" + "".join(
        f" {name + ' (deviation)':>28} |" for name in md_results)
    print(header)
    print("-" * len(header))
    for label, scale, unit in ([(l, 1000, "mA") for l in BOND_COLORS]
                               + [(l, 1, "deg") for l in ANGLE_COLORS]):
        ref = ref_bonds[label] if label in BOND_COLORS else ref_angles[label]
        row = f"{label:>10} | {ref.mean():10.4f} |"
        for name, (bonds, angles_, _) in md_results.items():
            values = bonds[label] if label in BOND_COLORS else angles_[label]
            delta = scale * (values.mean() - ref.mean())
            err = scale * blocked_error(values)
            row += f" {values.mean():9.4f}  ({delta:+6.1f} +- {err:4.1f} {unit}) |"
        print(row)
    print("\n(deviations from the CCSD(T) mean, with a blocked estimate of the MD "
          "statistical error)")


compare_structure({"zero-shot MD": (zs_bonds, zs_angles, ":")})

> **Your turn.** Which bond is the model most accurate on, and which is the worst? Is the
> disagreement large compared with the *width* of the reference distribution — i.e. would
> anyone notice it in a real simulation? Keep the numbers; §8 repeats this table with the
> fine-tuned model in a second column.

## 7. Fine-tune PET-MAD-XS

Everything so far used the model exactly as shipped. Now we specialise it.

**Fine-tuning** starts from the pretrained weights and continues training on our small
CCSD(T) set, with a small learning rate. The representation the model built from millions
of DFT structures — what a C-O bond looks like, how an environment maps to an energy — is
inherited in full; what gets adjusted is mostly the mapping onto *this* dataset's level of
theory. That is why a few hundred structures and a couple of minutes are enough, where
training from scratch would need orders of magnitude more of both.

`metatrain` is driven by a YAML options file, which we write from the notebook so that the
choices you made above (the split, the device) flow into it automatically. The parts worth
reading:

* `finetune: {read_from: ..., method: full}` — start from the PET-MAD-XS checkpoint and let
  **all** weights move. Cheaper alternatives exist (`lora`, or freezing everything except
  the last layer) and are worth exploring in the cookbook recipe linked at the end.
* `targets: energy: {..., forces: {key: forces}}` — train on energies *and* forces. Forces
  are 3N numbers per structure instead of one, so they carry far more information per
  expensive reference calculation.
* the model also refits its internal per-element energy baseline, which is how it absorbs
  the ~18 eV offset we had to fit by hand in §4.

> **Your turn.** `NUM_EPOCHS` below is the obvious knob. Run it once as-is, look at the
> learning curves in the next cell, and decide whether the run was too short, too long, or
> about right. (The learning rate is annealed to zero at `num_epochs`, so changing it
> changes the whole schedule, not just the stopping point.)

In [ ]:
NUM_EPOCHS = 30  # TODO: try 5, or 50, and watch the validation curves below
BATCH_SIZE = 16   # TODO: structures per gradient step
SEED = 0          # TODO: random seed for initial weights and batching

# `metatrain` runs as a separate process and reads its data from files, so the two splits
# are written into a scratch directory that disappears the moment training is over -- the
# lists in memory stay the only lasting copy. `finetuned.pt` and the `outputs/` logs are
# still written next to this notebook.
with tempfile.TemporaryDirectory() as scratch:
    train_xyz = Path(scratch) / "train.xyz"
    val_xyz = Path(scratch) / "val.xyz"
    options_path = Path(scratch) / "options.yaml"
    ase.io.write(train_xyz, train_frames)
    ase.io.write(val_xyz, val_frames)

    options_yaml = f'''
seed: {SEED}
device: {DEVICE}

architecture:
  name: pet
  training:
    finetune:
      read_from: {CKPT_PATH}
      method: full
    batch_size: {BATCH_SIZE}
    num_epochs: {NUM_EPOCHS}
    learning_rate: 1e-4
    checkpoint_interval: 100

training_set:
  systems:
    read_from: {train_xyz}
    length_unit: angstrom
  targets:
    energy:
      key: energy
      unit: eV
      forces:
        key: forces

validation_set:
  systems:
    read_from: {val_xyz}
    length_unit: angstrom
  targets:
    energy:
      key: energy
      unit: eV
      forces:
        key: forces

test_set: 0.0
'''
    options_path.write_text(options_yaml)

    t0 = time.time()
    subprocess.run(
        [sys.executable, "-m", "metatrain", "train", str(options_path), "-o", "finetuned.pt"],
        check=True,
    )
print(f"fine-tuning finished in {time.time() - t0:.1f} s")

`metatrain` logs every epoch to a CSV inside a timestamped `outputs/` directory. The two
things to look for: the **training** and **validation** curves should both fall, and they
should stay close to each other. If the validation error flattens out or turns upward
while the training error keeps dropping, the model has started memorising the training
structures instead of learning the physics — that is the moment to stop, or to add data.

In [ ]:
train_csv = sorted(Path("outputs").glob("*/*/train.csv"), key=lambda p: p.stat().st_mtime)[-1]
print(f"reading {train_csv}")
log = np.genfromtxt(train_csv, delimiter=",", skip_header=2)
epoch = log[:, 0]
train_e_mae, val_e_mae = log[:, 4], log[:, 9]
train_f_mae, val_f_mae = log[:, 5], log[:, 10]

fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=120)
axes[0].plot(epoch, train_e_mae, "o-", color="tab:blue", label="training")
axes[0].plot(epoch, val_e_mae, "o--", color="tab:cyan", label="validation")
axes[0].set(xlabel="epoch", ylabel="energy MAE (meV/atom)", yscale="log",
            title="Energy")
axes[0].legend()

axes[1].plot(epoch, train_f_mae, "s-", color="tab:orange", label="training")
axes[1].plot(epoch, val_f_mae, "s--", color="tab:red", label="validation")
axes[1].set(xlabel="epoch", ylabel=r"force MAE (meV/$\AA$)", yscale="log", title="Forces")
axes[1].legend()

fig.suptitle("Fine-tuning learning curves", fontsize=11)
fig.tight_layout()
plt.show()

## 8. How much did it help?

Same test set, same plots, same MD, same observables — only the calculator changes.

In [ ]:
finetuned_calculator = MetatomicCalculator("finetuned.pt", device=DEVICE)

ft_test_e, ft_test_f = predict(finetuned_calculator, test_frames, "fine-tuned test set")

# no energy shift this time: the fine-tuned model has learned the CCSD(T) reference itself
ft_energy_mae = 1000 * np.mean(np.abs(ft_test_e - ref_test_e)) / n_atoms
ft_forces_mae = 1000 * np.mean(np.abs(ft_test_f - ref_test_f))

print(f"residual energy offset: {np.mean(ref_test_e - ft_test_e) * 1000:.1f} meV "
      f"(it was {energy_shift * 1000:.0f} meV before fine-tuning)")
print(f"fine-tuned energy MAE: {ft_energy_mae:.2f} meV/atom  "
      f"(zero-shot: {zs_energy_mae:.2f})")
print(f"fine-tuned force  MAE: {ft_forces_mae:.1f} meV/A     "
      f"(zero-shot: {zs_forces_mae:.1f})")

parity_plots(ft_test_e, ft_test_f, "fine-tuned", "tab:orange", ft_energy_mae,
             ft_forces_mae)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5), dpi=120)
x = np.arange(2)
ax.bar(x - 0.2, [zs_energy_mae, zs_forces_mae], width=0.4, color="tab:blue",
       label="zero-shot")
ax.bar(x + 0.2, [ft_energy_mae, ft_forces_mae], width=0.4, color="tab:orange",
       label="fine-tuned")
for xi, (a, b) in enumerate([(zs_energy_mae, ft_energy_mae),
                             (zs_forces_mae, ft_forces_mae)]):
    ax.text(xi, max(a, b) * 1.15, f"{a / b:.1f}x better", ha="center", fontsize=9)
ax.set(xticks=x, yscale="log",
       ylabel="mean absolute error (log scale)",
       title="Test-set accuracy before and after fine-tuning")
ax.set_xticklabels(["energy\n(meV/atom)", "forces\n(meV/$\\AA$)"])
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
ft_relaxed = relax(frames[0], finetuned_calculator)

t0 = time.time()
ft_traj, ft_kinetic, ft_potential = run_md(
    finetuned_calculator, ft_relaxed, nsteps=MD_STEPS+EQUILIBRATION, dt_fs=MD_TIMESTEP_FS,
    temperature_K=MD_TEMPERATURE_K, desc="fine-tuned MD")
print(f"fine-tuned MD: {len(ft_traj)} frames in {time.time() - t0:.1f} s")

plot_md_diagnostics(ft_kinetic, ft_potential, MD_TIMESTEP_FS, "fine-tuned")

ft_bonds = bond_lengths(ft_traj[EQUILIBRATION:])
ft_angles = bond_angles(ft_traj[EQUILIBRATION:])

compare_structure({"zero-shot MD": (zs_bonds, zs_angles, ":"),
                   "fine-tuned MD": (ft_bonds, ft_angles, "-")})

> **Your turn.** Look at the table — and at the error bars before concluding anything.
> Which deviations moved by more than the statistical uncertainty of a 2 ps run?
>
> You will probably find that the three bond lengths hardly move at all: fine-tuning, which
> improved the forces by a factor of five, changes them by a few thousandths of an Angstrom
> and does not systematically improve them. That is the first lesson, and it is a general
> one — bond lengths and angles are *stiff* coordinates, pinned down by steep restoring
> forces, so a good potential gets them roughly right and a much better potential
> can only get them slightly righter. Improving a model does not improve every observable
> in proportion; it improves the **soft** ones, the ones governed by small energy
> differences.
>
> You may also find one angle shifting by several degrees, well outside its quoted error.
> Before believing that number, think about what the error bar does *not* know: 2 ps
> contains only a handful of hydroxyl rotations, and any coordinate coupled to that slow
> torsion inherits its poor sampling, which block averaging cannot see. Rerun with a
> different `seed` to test it. And note where it points — the two models disagree about
> the torsion itself, which is exactly the subject of §9.

## 9. The hard test: conformer energetics

<img src="pics/anti-gauche.png" width=600 height=250 />

Ethanol can rotate its hydroxyl group about the C2-O bond. Two shapes come out of that
rotation, distinguished by the **C1-C2-O-H dihedral angle** $\phi$:

* **anti** (also called *trans*), $\phi = 180^\circ$ — the O-H bond points away from the
  methyl group;
* **gauche**, $\phi \approx \pm 60^\circ$ — the O-H bond is rotated to one side. There are
  **two** of these, mirror images of each other, and they are exactly equal in energy.

They are not different molecules: they interconvert many times a nanosecond at room
temperature. But which one is lower in energy, and by how much, decides the equilibrium
populations, and analogous questions decide protein side-chain packing, drug binding
poses, and the melting point of a molecular crystal.

Here is what makes this a *hard* test. The anti/gauche energy difference in ethanol is
known from microwave spectroscopy and high-level quantum chemistry to be **at most a few
meV** — roughly 0.5 kJ/mol, with anti usually found marginally lower. Compare that with the
numbers we have been printing all notebook: a zero-shot energy error of several meV per
atom, and total energies of about $-31$ eV. We are asking the model to resolve one part in
$10^4$ of the quantity it predicts. Nothing we have measured so far tells us whether it
can.

### 9.1 What the reference data already knows

Before touching the models: the reference dataset is a thermal sample, so it visits all
three wells. Histogramming its dihedral angle gives the *populations*, which is the
physically observable version of this question.

In [ ]:
ref_dihedral = np.array([a.get_dihedral(*TORSION) for a in frames])

fig, ax = plt.subplots(figsize=(7, 3.5), dpi=120)
ax.hist(ref_dihedral, bins=72, range=(0, 360), color="tab:grey")
for phi, name in [(60, "gauche +"), (180, "anti"), (300, "gauche -")]:
    ax.axvline(phi, color="tab:red", ls="--", lw=1)
    ax.text(phi, ax.get_ylim()[1] * 0.92, name, ha="center", fontsize=9, color="tab:red")
ax.set(xlabel=r"C1-C2-O-H dihedral $\phi$ (degrees)", ylabel="structures",
       xticks=np.arange(0, 361, 60), title="Where the CCSD(T) dataset actually sits")
fig.tight_layout()
plt.show()

# basin boundaries put where the barriers are (see the scan in 9.2), so that each
# well gets an equal share of the circle and the comparison is not biased by binning
basins = {"gauche +": (0, 120), "anti": (120, 240), "gauche -": (240, 360)}
counts = {name: int(((ref_dihedral > lo) & (ref_dihedral < hi)).sum())
          for name, (lo, hi) in basins.items()}
n_gauche = counts["gauche +"] + counts["gauche -"]
print(counts)
print(f"gauche : anti population ratio = {n_gauche / counts['anti']:.2f} : 1")

Two features are worth pausing on.

The dataset spends roughly twice as much time **gauche** as **anti** — and that is not yet
evidence that gauche is lower in energy, because there are *two* gauche wells and only one
anti well. A 2:1 ratio is what pure degeneracy gives you even if the two forms have
identical energy, and the measured ratio is close to it. In free-energy terms the
degeneracy is worth $k_BT\ln 2 \approx 30$ meV at the 500 K this data was sampled at,
which is far *larger* than the energy difference we are chasing: entropy settles this
contest before energetics gets a vote.

Second, the wells are broad and the barriers between them are clearly crossed. That is what
lets a thermal dataset teach a model about the whole torsional profile — but it also means
we cannot read the profile off the histogram cleanly, because every other vibration is
excited at the same time. For a clean answer we have to ask the models directly.

### 9.2 The relaxed torsional potential

The standard construction is a **relaxed scan**: fix the dihedral $\phi$ at a series of
values and, at each one, minimise the energy over *all other* degrees of freedom. What comes
out, $E(\phi)$, is the effective potential governing the rotation, with every other
coordinate free to adapt.

Note that each point is a separate constrained minimisation started from the same relaxed
geometry, so there is no hysteresis: the scan does not depend on which direction we walk in.

> **Your turn.** Predict, before running: how many minima will the curve have, and where?
> Will the curve be symmetric about $\phi = 180^\circ$, and why? And — the real question —
> do you expect the zero-shot and fine-tuned models to give the *same* answer, given that
> §8 showed they differ by only ~20 meV/A in forces?

In [ ]:
from ase.constraints import FixInternals

SCAN_ANGLES = np.arange(0, 360, 10.0)  # TODO: 5.0 for a finer (2x slower) scan


def torsion_scan(calculator, start_atoms, angles_deg, fmax=0.0001):
    # relaxed scan of the C1-C2-O-H dihedral: at each angle, relax everything else
    base = start_atoms.copy()
    base.calc = calculator
    LBFGS(base, logfile=None).run(fmax=fmax, steps=600)

    energies = []
    for phi in tqdm(angles_deg, desc="relaxed torsion scan", leave=False):
        atoms = base.copy()
        atoms.set_dihedral(*TORSION, phi)
        atoms.set_constraint(FixInternals(dihedrals_deg=[[phi, list(TORSION)]]))
        atoms.calc = calculator
        LBFGS(atoms, logfile=None).run(fmax=fmax, steps=600)
        energies.append(atoms.get_potential_energy())
    return np.array(energies)


t0 = time.time()
zs_scan = torsion_scan(zero_shot_calculator, zs_relaxed, SCAN_ANGLES)
ft_scan = torsion_scan(finetuned_calculator, ft_relaxed, SCAN_ANGLES)
print(f"two scans of {len(SCAN_ANGLES)} points each in {time.time() - t0:.1f} s")

In [ ]:
anti_index = int(np.argmin(np.abs(SCAN_ANGLES - 180)))


def relative_to_anti(energies):
    # meV, measured from the anti conformer of that same model
    return 1000 * (energies - energies[anti_index])


def mirror_average(profile):
    # phi and 360 - phi are mirror images and must have identical energy
    return 0.5 * (profile + profile[(-np.arange(len(profile))) % len(profile)])


zs_profile, ft_profile = relative_to_anti(zs_scan), relative_to_anti(ft_scan)

fig, ax = plt.subplots(figsize=(8.5, 5.4), dpi=120)
for profile, name, color in [(zs_profile, "PET-MAD-XS zero-shot", "tab:blue"),
                             (ft_profile, "fine-tuned on CCSD(T)", "tab:orange")]:
    ax.plot(SCAN_ANGLES, profile, "o-", color=color, ms=4, lw=1.5, label=name)
    ax.plot(SCAN_ANGLES, mirror_average(profile), "--", color=color, lw=1, alpha=0.7,
            label=f"{name}, mirror-averaged")

ax.axhline(0, color="k", lw=0.8)
for phi, name in [(60, "gauche +"), (180, "anti"), (300, "gauche -")]:
    ax.axvline(phi, color="grey", ls=":", lw=0.8)
    ax.text(phi, ax.get_ylim()[1] * 0.95, name, ha="center", fontsize=8, color="grey")
ax.set(xlabel=r"C1-C2-O-H dihedral $\phi$ (degrees)",
       ylabel="energy relative to the anti conformer (meV)",
       xticks=np.arange(0, 361, 60),
       title="Comparison of the C1-C2-O-H torsional potential")
# legend below the axes, so it cannot sit on top of the conformer labels
ax.legend(fontsize=8, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.16))
fig.tight_layout()
plt.show()

In [ ]:
def profile_summary(profile, name):
    sym = mirror_average(profile)
    left = (SCAN_ANGLES > 20) & (SCAN_ANGLES < 140)
    gauche_index = int(np.where(left)[0][np.argmin(sym[left])])
    between = slice(min(gauche_index, anti_index), max(gauche_index, anti_index) + 1)
    print(f"{name}")
    print(f"   gauche well at phi = {SCAN_ANGLES[gauche_index]:.0f} deg")
    print(f"   E(gauche) - E(anti)      = {sym[gauche_index]:+7.1f} meV "
          f"({sym[gauche_index] * 0.0230604 :+.2f} kcal/mol)")
    print(f"   anti -> gauche barrier   = {sym[between].max() - 0:7.1f} meV"
          f"({(sym[between].max() - 0) * 0.0230604 :+.2f} kcal/mol)")
    print(f"   cis barrier (phi = 0)    = {sym[0]:7.1f} meV"
          f"({sym[0] * 0.0230604 :+.2f} kcal/mol)")
    print(f"   mirror-symmetry breaking = {np.abs(profile - mirror_average(profile)).max():7.1f} meV "
          f"(should be 0)")
    return sym[gauche_index]


zs_delta = profile_summary(zs_profile, "PET-MAD-XS zero-shot")
ft_delta = profile_summary(ft_profile, "fine-tuned on CCSD(T)")
print(f"\nthe two models disagree about E(gauche) - E(anti) by "
      f"{abs(zs_delta - ft_delta):.1f} meV")

The two curves have the same *shape* — three wells, a high barrier at the eclipsed
$\phi = 0^\circ$ geometry.

The zero-shot model places the gauche form clearly above anti, by an amount far larger than
the few meV the experiment allows. Fine-tuning collapses that gap to a handful of meV, and
may even invert the ordering.

There is a second, subtler thing in the plot. The raw curves are **not exactly symmetric**
about $180^\circ$, even though $\phi$ and $360^\circ - \phi$ are mirror images of one
another and must have *exactly* the same energy; the dashed curves show what each model
would say if it respected that symmetry.

> **Your turn.** Read the barrier heights off the printout and compare them with
> $k_BT \approx 43$ meV at the 500 K our trajectories ran at. Is the anti $\to$ gauche
> barrier something the molecule crosses often, rarely, or never in a 2 ps run? Predict
> first, then check against the trace below.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5), dpi=120)
for traj, name, color in [(zs_traj[EQUILIBRATION:], "zero-shot MD", "tab:blue"),
                          (ft_traj[EQUILIBRATION:], "fine-tuned MD", "tab:orange")]:
    phi = np.array([a.get_dihedral(*TORSION) for a in traj])
    ax.plot(np.arange(len(phi)) * MD_TIMESTEP_FS / 1000, phi, lw=0.7, color=color,
            label=name, alpha=0.8)
ax.set(xlabel="time (ps)", ylabel=r"$\phi$ (degrees)", yticks=np.arange(0, 361, 60),
       ylim=(0, 360), title="Does the hydroxyl group actually rotate during our MD?")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

### 9.3 An independent check that does not rely on the scan

The scan compares the two models with each other, but it has no CCSD(T) reference: nobody
has run coupled cluster on those particular constrained geometries.

There is a way around that. For the 2,000 dataset structures we *do* have exact CCSD(T)
energies, so we can look at the **residual**, $E_\text{model} - E_\text{CCSD(T)}$, with the
constant offset removed, and ask how it varies with $\phi$. Because both energies refer to
the same geometry, everything that is not a model error — thermal excitation of the bonds,
the angles, the methyl rotation — cancels exactly.

If the residual is flat in $\phi$, the model reproduces the torsional energetics however
wrong it may be overall. If the residual is *tilted*, the tilt is precisely the error the
model makes on the conformer energy difference.

In [ ]:
N_RESIDUAL = 600  # TODO: raise for smoother curves, at ~25 structures/second per model
sub_idx = np.random.default_rng(1).choice(len(frames), N_RESIDUAL, replace=False)
sub_frames = [frames[i] for i in sub_idx]
sub_dihedral = ref_dihedral[sub_idx]
sub_ref_e = np.array([a.get_potential_energy() for a in sub_frames])

residuals = {}
for name, calculator in [("zero-shot", zero_shot_calculator),
                         ("fine-tuned", finetuned_calculator)]:
    pred_e, _ = predict(calculator, sub_frames, f"{name}: residual scan")
    r = pred_e - sub_ref_e
    residuals[name] = 1000 * (r - r.mean())  # meV, constant offset removed

bin_edges = np.arange(0, 361, 20)
bin_centres = 0.5 * (bin_edges[1:] + bin_edges[:-1])

fig, ax = plt.subplots(figsize=(8.5, 4.5), dpi=120)
for (name, r), color in zip(residuals.items(), ["tab:blue", "tab:orange"]):
    means, errors = [], []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        in_bin = (sub_dihedral >= lo) & (sub_dihedral < hi)
        means.append(r[in_bin].mean() if in_bin.sum() else np.nan)
        errors.append(r[in_bin].std() / np.sqrt(max(in_bin.sum(), 1)))
    ax.errorbar(bin_centres, means, yerr=errors, fmt="o-", color=color, capsize=3,
                label=f"{name} (RMSE {np.std(r) / n_atoms:.2f} meV/atom)")

ax.axhline(0, color="k", lw=0.8)
for phi in (60, 180, 300):
    ax.axvline(phi, color="grey", ls=":", lw=0.8)
ax.set(xlabel=r"C1-C2-O-H dihedral $\phi$ (degrees)",
       ylabel=r"$E_\mathrm{model} - E_\mathrm{CCSD(T)}$ (meV)",
       xticks=np.arange(0, 361, 60),
       title="Model error as a function of conformation (flat = no conformational bias)")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

print(f"{'model':>12} | {'anti':>8} | {'gauche':>8} | error on E(gauche) - E(anti)")
print("-" * 66)
for name, r in residuals.items():
    anti_mask = (sub_dihedral > 120) & (sub_dihedral < 240)  # basins split at the barriers
    gauche_mask = ~anti_mask
    bias = r[gauche_mask].mean() - r[anti_mask].mean()
    print(f"{name:>12} | {r[anti_mask].mean():+7.1f} meV | {r[gauche_mask].mean():+7.1f} meV "
          f"| {bias:+7.1f} meV")

This is the same story told by completely different structures — thermally distorted
snapshots instead of constrained minima — and it points the same way: the zero-shot model
carries a systematic bias of tens of meV between the anti and gauche regions, in the same
direction as the gap in its scan (+28 meV here, against the scan's +46). The two numbers are
not expected to be identical — one is a bias averaged over a thermal ensemble, the other a
difference between two relaxed minima — but agreeing in sign and magnitude is the point: the
discrepancy in §9.2 is not an artefact of how we ran the scan, it is a real property of the
model. Fine-tuning cuts the bias to a couple of meV.

Subtracting the measured bias from each model's scan gives an implied CCSD(T) answer of
roughly +10 to +20 meV in favour of anti. Both routes therefore agree on *which* conformer is
more stable, and that agrees with spectroscopy — but our margin is well above the ~5 meV that
experiment supports, and, as the next section shows, well above what we can honestly claim to
resolve.

### 9.4 What about equivariance?

Some operations must leave the energy of a molecule **exactly** unchanged. **Rotating** it is
the obvious one: space has no preferred direction, so any dependence of the energy on how the
molecule happens to be oriented in the box is reporting something that does not exist.
**Reflecting** it in a mirror is another — parity is an exact symmetry of the non-relativistic
electronic Hamiltonian, and it is the operation that turns gauche+ into gauche-, which is why
those two wells must be exactly degenerate and why we mirror-averaged the scan back in §9.2.

PET is an *unconstrained* architecture: instead of hard-wiring these symmetries into its
functional form, it learns them from data, so it satisfies them only approximately. That
gives us a free measurement. Take any structure, rotate it, and re-evaluate: the answer must
be zero, so whatever the model reports instead is pure error — and, crucially, we needed no
reference calculation whatsoever to know the right answer.

In [ ]:
def invariance_errors(calculator, atoms_list, name):
    # rotating a molecule cannot change its energy: whatever the model
    # reports for these two differences is pure error, measured with no reference data
    rng_local = np.random.default_rng(0)
    rotation = []
    for atoms in tqdm(atoms_list, desc=f"{name}: invariance test", leave=False):
        original = atoms.copy()
        original.calc = calculator
        e0 = original.get_potential_energy()

        rotated = atoms.copy()
        rotated.rotate(rng_local.uniform(0, 360), rng_local.normal(size=3))
        rotated.calc = calculator
        rotation.append(1000 * (rotated.get_potential_energy() - e0)/len(atoms))  # meV/atom

    rotation = np.array(rotation)
    print(f"{name:>11}:  rotation RMS {np.sqrt((rotation ** 2).mean()):5.2f} meV/atom,"
          f" MAE  {np.abs(rotation).mean():5.2f} meV/atom  |")
    return rotation


invariance = {
    name: invariance_errors(calc, frames[::20], name)
    for name, calc in [
        ("zero-shot", zero_shot_calculator),
        ("fine-tuned", finetuned_calculator),
    ]
}

fig, ax = plt.subplots(figsize=(6.5, 3.6), dpi=120)

for (name, result), color in zip(invariance.items(),["tab:blue", "tab:orange"]):

    ax.hist(result,bins=15,alpha=0.6,color=color,label=name,density=True)

ax.axvline(0, color="k", lw=1.2)

ax.set_xlabel(r"$E(\mathrm{rotated}) - E(\mathrm{original})$ (meV/atom)")
ax.set_ylabel("Structures")
ax.set_title("Rotational invariance error")
ax.legend(fontsize=8)

fig.tight_layout()
plt.show()

The error comes out at about 1 meV/atom zero-shot, shrinking several-fold after fine-tuning:
the model's answer depends slightly on **which way the molecule happens to be pointing**. That
is the cleanest possible picture of what "learned, not enforced, symmetry" means.

The conclusion is therefore:
* the zero-shot model gets structural quantities such as bond lengths and angles right, and the forces are good enough to run MD at the reference temperature;
* the zero-shot model gets the conformer energetics **qualitatively wrong**, by tens of
  meV, and this is invisible in every diagnostic from §4 to §8;
* fine-tuning on a few hundred CCSD(T) structures removes most of that bias, bringing the
  prediction into agreement with experiment **within its own uncertainty**;
* the fine-tuned model does resolve the *sign* of the anti/gauche difference — but only just.
  Its rotational error of ~0.55 meV/atom is about 5 meV for a 9-atom molecule, against a
  predicted gap of ~8 meV. Report the sign; do not report the value to three digits.


> **Your turn, to finish.** Everything in §9 was run on a model fine-tuned with your choice
> of `n_train` and `NUM_EPOCHS`. Go back to §2, change them, and come back: does more data
> flatten the residual curve further? Does it shrink the invariance errors? Which of the two
> matters more for getting the conformer question right?

## Where to go next

* [PET-MAD tutorial](https://atomistic-cookbook.org/examples/pet-mad/pet-mad.html) — how to
  set up and run PET-MAD yourself with ASE, i-PI and LAMMPS.
* [Fine-tuning PET-MAD](https://atomistic-cookbook.org/examples/pet-finetuning/pet-ft.html)
  — a deeper dive into fine-tuning strategies (full, last-layer, LoRA), and how to choose
  between them.
* [Uncertainty quantification with PET-MAD](https://atomistic-cookbook.org/examples/pet-mad-uq/pet-mad-uq.html)
  — the upstream PET-MAD checkpoints ship with a built-in uncertainty estimator (LLPR); the
  one in this repository does not carry it, since it roughly doubles the cost of every
  prediction. Section 9.4 is a hand-rolled substitute for what it does properly.
* The companion `water-md` notebook — the same foundation model used zero-shot for
  condensed-phase molecular dynamics in LAMMPS, where fine-tuning is not an option because
  there is no reference data to fine-tune on.